## Essay Writer

### Defining the AgentState and the Prompts

In [1]:
# Loading keys
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [3]:
# Standard imports
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini', temperature=0)


In [5]:
from typing import TypedDict, List, Annotated

# Creating a State
class AgentState(TypedDict):
    task: str # Human message or prompt about the subject of the essay.
    plan: str # Plan about how the task will be performed.
    draft: str # Draft essays for the subject mentioend in the task
    critique: str # Critique response of the critique agent for the essay.
    content: List[str] # List of documents from Tavily Search AI for helping to write the essay.
    revision_number: int # The current revision number
    max_revisions: int # Maximum allowed revisions.

In [6]:
# Defining the planning prompt
PLAN_PROMPT = '''You are an expert writer tasked with writing a high level outline of an essay.
Write such an outline for the user provided topic.
Give an outline of the essay along with any relevant notes or instructions for the sections.'''

# Research agent prompt which will reasearch using Tavily to find resources to write this essay.
RESEARCH_PLAN_PROMPT = '''You are a researcher charged with providing information that can be used when writing the following essay. 
Generate a list of seach queries that will gather any relevant information. Only generate 3 queries max.'''

# Define the Writer prompt
WRITER_PROMPT = '''You are an essay assistant tasked with writing excellent 5-paragraph essays.
Generate the best essay possible for the user's request and the initial outline.
If the user provides critique, respond with a revised version of your previous attempts.
Utilize all the information below as needed:

--------

{content}'''

# Reflection Prompt
REFLECTION_PROMPT = '''You are a teacher grading an essay submission.
Generate critique and recommendations for the user's submission.
Provide detailed recommendations, including requests for length, depth, style, etc.'''

# Research Critique prompt
# Tavili research agent
RESEARCH_CRITIQUE_PROMPT = '''You are a researcher charged with providing information that can be used when asking any requested revisions (as outlined below).
Generate a list of seach queries that will gather any relevant information. Only generate 3 queries max.'''

In [9]:
# To generate the queries to pass to Tavily we will use function calling to make sure we get a list of strings the LLM
# To do this we will use a pydantic model to get back the results.

from pydantic import BaseModel

class Queries(BaseModel):
    queries: List[str]

In [12]:
# Instantiating the Tavily client

from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])